#### Modelo extrator de texto de PDF

##### Biblioteca

In [47]:
import pandas as pd
import fitz
import glob
from tqdm import tqdm
import os
import time

import pytesseract
from PIL import Image

import logging

import json
import logging
import time
from pathlib import Path

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    PdfPipelineOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.models.ocr_mac_model import OcrMacOptions
from docling.models.tesseract_ocr_cli_model import TesseractCliOcrOptions
from docling.models.tesseract_ocr_model import TesseractOcrOptions

from docling.document_converter import DocumentConverter
from tqdm import tqdm


In [ ]:
torch.cuda.is_available()

In [49]:
import torch
torch.cuda.is_available()

False

##### Modelo

In [45]:
def extrai_texto_pdf_LEGIVEL(pdf_file, tipo_doc):
    ini = time.time()
    try:
        # extrai metadados
        numero_tj = pdf_file.split("\\")[1].split("_")[0]
        ### para ver amanha
        #md_text = pymupdf4llm.to_markdown(r"C:\Estudos e Projetos\2024-LLM-Agronomo-Virtual\pdf\0000444-08.2022.8.26.0052_relatório_final_pags_73-77.pdf")
        

        # Abrir o PDF com PyMuPDF
        pdf_document = fitz.open(pdf_file)
        # Extrai o total de páginas
        total_paginas = int(pdf_document.page_count)
        # Extrair texto de cada página do PDF
        texto_total = ''
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            # Ajusta o valor do retângulo conforme necessário
            if tipo_doc == 'sentença':
                rect = fitz.Rect([30, 130, 570, 1000])
            elif tipo_doc == 'decisão':
                rect = fitz.Rect([10, 190, 550, 810])
            elif tipo_doc == 'inquérito policial':
                rect = fitz.Rect([30, 200, 572, 900])
            elif tipo_doc == 'laudo iml':
                rect = fitz.Rect([30, 140, 548, 900])
            elif tipo_doc == 'laudo médico':
                rect = fitz.Rect([30, 140, 560, 900])
            elif tipo_doc == 'laudo médico (digitalizado)':
                rect = fitz.Rect([30, 140, 530, 900])
            else:
                rect = fitz.Rect([30, 20, 572, 1000])  # Definindo o retângulo para extração de texto
            texto_total += page.get_textbox(rect)  # Extrai o texto completo da página
            #print(texto_total)
            #print(pdf_document.name)
        tempo_exec = time.time() - ini
        #print([texto_total,total_paginas,fim]) # print para debug
        return numero_tj, tipo_doc, texto_total, total_paginas, tempo_exec # Retorna o texto extraído
    
    except Exception as e:
        #print(f"Erro ao extrair texto do PDF {pdf_file}: {e}") bloqueei isso, por que tem muitas imagens.
        return None



def extrai_texto_de_imagem_OCR(image):
    try:
        # Se a imagem for colorida, tente não fazer a conversão para 'L' e o redimensionamento.
        bw_image = image.convert("L")  # Isso pode ser removido para testar sem conversão para preto e branco
        #bw_image = bw_image.resize((bw_image.width // 2, bw_image.height // 2))  # Isso pode ser removido para testar sem redimensionamento

        # Verificar se a imagem está vazia
        if bw_image.size == (0, 0):
            raise ValueError("Imagem convertida está vazia")
        
        # Salvar a imagem temporariamente para processamento
        temp_image_path = Path("C:/Estudos e Projetos/2024-LLM-Agronomo-Virtual/1 - criação_base/pasta_docs_tmp/temp_image.png")
        bw_image.save(temp_image_path)
        
        # Definindo as opções do pipeline de OCR
        pipeline_options = PdfPipelineOptions()
        pipeline_options.do_ocr = True
        pipeline_options.do_table_structure = True
        pipeline_options.table_structure_options.do_cell_matching = True
        pipeline_options.ocr_options.lang = ["pt-br"]
        pipeline_options.accelerator_options = AcceleratorOptions(device=AcceleratorDevice.CPU)  # Configuração correta para usar CUDA

        # Criar o conversor de documentos com as opções configuradas
        converter = DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
        )

        # Converter a imagem temporária para texto
        doc = converter.convert(temp_image_path).document
        md = doc.export_to_markdown()

        # Remover o arquivo temporário
        temp_image_path.unlink()

        return md

    except Exception as e:
        print(f"Erro ao converter imagem para texto: {e}")
        return None


def extrai_texto_OCR(item, tipo_doc):   
    """
    Iterates over a list of source items, extracts metadata, opens PDF documents, and processes each page as a black and white image with 300 DPI.
    Args:
        source (list): List of file paths to PDF documents.
    Variables:
        lista_md (list): List to store metadata and extracted text.
        numero_tj (str): Extracted metadata representing the document number.
        tipo_doc (str): Extracted metadata representing the document type.
        pdf_document (fitz.Document): PDF document object.
        md_blocks (list): List to store markdown text blocks for each page.
        page (fitz.Page): Page object of the PDF document.
        pix (fitz.Pixmap): Pixmap object representing the page image.
        image (PIL.Image): Image object created from the pixmap.
        bw_image (PIL.Image): Black and white version of the image.
    Note:
        The code for extracting text and converting it to markdown is commented out.
        The code for appending the markdown text to the list and creating a DataFrame is also commented out.
    """
    # Cria a a variável para armazenar o tempo inicial
    ini = time.time()
    # extrai metadados
    numero_tj = item.split("\\")[1].split("_")[0]
    # Abrir o documento PDF
    pdf_document = fitz.open(item.replace("\\", "/"))
    # Extrai o total de páginas
    total_paginas = int(pdf_document.page_count)
    # Salvar cada página como uma imagem em preto e branco com 300 DPI
    md_extract = ''
    for i in tqdm(range(len(pdf_document))):
        # faz a leitura da página do documento
        page = pdf_document.load_page(i)
        # Ajusta o valor do retângulo conforme necessário
        if tipo_doc == 'sentença':
            rect = fitz.Rect([30, 130, 570, 1000])
        elif tipo_doc == 'decisão':
            rect = fitz.Rect([10, 190, 550, 810])
        elif tipo_doc == 'inquérito policial':
            rect = fitz.Rect([30, 200, 572, 900])
        elif tipo_doc == 'laudo iml':
            rect = fitz.Rect([30, 140, 548, 900])
        elif tipo_doc == 'laudo médico':
            rect = fitz.Rect([30, 140, 560, 900])
        elif tipo_doc == 'laudo médico (digitalizado)':
            rect = fitz.Rect([30, 140, 530, 900])
        else:
            rect = fitz.Rect([30, 20, 572, 1000])  # Definindo o retângulo para extração de texto
        # pega a imagem da página em 300 DPI
        pix = page.get_pixmap(dpi=250, clip=rect)
        # converte a imagem em um objeto PIL
        image = pix.pil_image() # Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        # extrai texto e converte em markdown
        md_extract += extrai_texto_de_imagem_OCR(image)
        # salva o tempo final
        fim = time.time() - ini
        # cria a variavel do tempo de execução
        tempo_exec = fim - ini
    return numero_tj, tipo_doc, md_extract, total_paginas, tempo_exec # Retorna o texto extraído


#### Execução do modelo - OCR

In [46]:

# Lista de tipos de documentos
tipos = [
    "decisão",
    "boletim de ocorrência",
    "inquérito policial",
    "laudo iml",
    "laudo médico",
    "laudo médico (digitalizado)",
    "laudo pericial",
    "laudo pericial (digitalizado)",
    "relatório de investigações",
    "relatório final",
    "relatório final digitalizado",
    "sentença"
]
# Iterando por cada tipo de documento
for pasta in tipos:
    # Pegando os primeiros 5 arquivos PDF de cada tipo de pasta
    source = glob.glob(f"C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/{pasta}/*.pdf")
    for item in source:
        tipo_doc = pasta
        txt = []  # Lista para armazenar os dados extraídos
        # Process the PDF
        extrai_texto_OCR(item, tipo_doc)

  0%|          | 0/1 [00:00<?, ?it/s]

2025-02-04 18:13:12,193 - INFO - Going to convert document batch...
2025-02-04 18:13:12,421 - INFO - Accelerator device: 'cpu'
2025-02-04 18:13:13,530 - INFO - Accelerator device: 'cpu'
2025-02-04 18:13:13,667 - INFO - Accelerator device: 'cpu'
2025-02-04 18:13:13,855 - INFO - Processing document temp_image.png
  0%|          | 0/1 [00:04<?, ?it/s]


KeyboardInterrupt: 

In [36]:
import os
import glob
import pandas as pd  # Certifique-se de importar o pandas para usar DataFrame

# Lista de tipos de documentos
tipos = [
    "decisão",
    "boletim de ocorrência",
    "inquérito policial",
    "laudo iml",
    "laudo médico",
    "laudo médico (digitalizado)",
    "laudo pericial",
    "laudo pericial (digitalizado)",
    "relatório de investigações",
    "relatório final",
    "relatório final digitalizado",
    "sentença"
]

# Iterando por cada tipo de documento
for pasta in tipos:
    # Pegando os primeiros 5 arquivos PDF de cada tipo de pasta
    source = glob.glob(f"C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/{pasta}/*.pdf")
    tipo_doc = pasta
    dados = []  # Lista para armazenar os dados extraídos

    # Limitando aos primeiros 5 arquivos, se houver
    source = source[:5]

    for pdf_file in source:
        # Verificando se o arquivo existe
        if not os.path.exists(pdf_file):
            print(f"Arquivo não encontrado: {pdf_file}")
            continue

        print(f"Processando o arquivo: {pdf_file}")
        try:
            # Assegurando que processa_pdf retorne sempre uma lista (mesmo vazia)
            extracted_data = processa_pdf(pdf_file, tipo_doc)
            if extracted_data:  # Se algum dado for extraído, adiciona à lista
                dados += extracted_data
        except Exception as e:
            print(f"Erro ao processar o PDF {pdf_file}: {e}")

    # Salvar os dados na planilha de origem dados
    if dados:  # Verifica se a lista de dados não está vazia
        # Convertendo para DataFrame
        dados_df = pd.DataFrame(dados, columns=['numero_tj', 'tipo_doc', 'texto_total', 'total_paginas', 'tempo_exec', 'endereco_doc'])
        # Salvando em um arquivo Excel
        output_path = os.path.join('C:/Estudos e Projetos/2024-LLM-Agronomo-Virtual/1 - criação_base/pasta_raspagem_teste/pasta_raspagem_ocr', 
                                   f'lista_raspagem_pdf_ocr_{tipo_doc}.xlsx')
        dados_df.to_excel(output_path, index=False)
        print(f"Dados salvos para o tipo {tipo_doc} em {output_path}")
    else:
        print(f"Nenhum dado extraído para o tipo {tipo_doc}.")


Processando o arquivo: C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/decisão\0000008-83.2021.8.26.0052_decisão_unificado.pdf


Processando PDFs:   0%|          | 0/161 [00:00<?, ?it/s]2025-02-04 18:00:44,402 - WARNING - Arquivo não encontrado: C
2025-02-04 18:00:44,403 - WARNING - Arquivo não encontrado: :
2025-02-04 18:00:44,404 - ERROR - Erro ao tentar abrir o PDF /: cannot unpack non-iterable NoneType object
2025-02-04 18:00:44,404 - WARNING - Arquivo não encontrado: E
2025-02-04 18:00:44,406 - WARNING - Arquivo não encontrado: s
2025-02-04 18:00:44,406 - WARNING - Arquivo não encontrado: t
2025-02-04 18:00:44,407 - WARNING - Arquivo não encontrado: u
2025-02-04 18:00:44,407 - WARNING - Arquivo não encontrado: d
2025-02-04 18:00:44,409 - WARNING - Arquivo não encontrado: o
2025-02-04 18:00:44,409 - WARNING - Arquivo não encontrado: s
2025-02-04 18:00:44,410 - WARNING - Arquivo não encontrado:  
2025-02-04 18:00:44,410 - WARNING - Arquivo não encontrado: e
2025-02-04 18:00:44,411 - WARNING - Arquivo não encontrado:  
2025-02-04 18:00:44,411 - WARNING - Arquivo não encontrado: P
2025-02-04 18:00:44,412 - WARN

Processando o arquivo: C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/decisão\0000037-31.2024.8.26.0052_decisão_unificado.pdf


Processando PDFs:   0%|          | 0/161 [00:00<?, ?it/s]2025-02-04 18:00:44,472 - WARNING - Arquivo não encontrado: C
2025-02-04 18:00:44,472 - WARNING - Arquivo não encontrado: :
2025-02-04 18:00:44,472 - ERROR - Erro ao tentar abrir o PDF /: cannot unpack non-iterable NoneType object
2025-02-04 18:00:44,473 - WARNING - Arquivo não encontrado: E
2025-02-04 18:00:44,473 - WARNING - Arquivo não encontrado: s
2025-02-04 18:00:44,473 - WARNING - Arquivo não encontrado: t
2025-02-04 18:00:44,474 - WARNING - Arquivo não encontrado: u
2025-02-04 18:00:44,474 - WARNING - Arquivo não encontrado: d
2025-02-04 18:00:44,474 - WARNING - Arquivo não encontrado: o
2025-02-04 18:00:44,474 - WARNING - Arquivo não encontrado: s
2025-02-04 18:00:44,476 - WARNING - Arquivo não encontrado:  
2025-02-04 18:00:44,476 - WARNING - Arquivo não encontrado: e
2025-02-04 18:00:44,476 - WARNING - Arquivo não encontrado:  
2025-02-04 18:00:44,477 - WARNING - Arquivo não encontrado: P
2025-02-04 18:00:44,477 - WARN

Processando o arquivo: C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/decisão\0000038-16.2024.8.26.0052_decisão_unificado.pdf


...

Processando o arquivo: C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/sentença\0000074-68.2018.8.26.0537_sentença_unificado.pdf


Processando PDFs:   0%|          | 0/163 [00:00<?, ?it/s]2025-02-04 18:00:47,784 - WARNING - Arquivo não encontrado: C
2025-02-04 18:00:47,784 - WARNING - Arquivo não encontrado: :
2025-02-04 18:00:47,784 - ERROR - Erro ao tentar abrir o PDF /: cannot unpack non-iterable NoneType object
2025-02-04 18:00:47,784 - WARNING - Arquivo não encontrado: E
2025-02-04 18:00:47,785 - WARNING - Arquivo não encontrado: s
2025-02-04 18:00:47,786 - WARNING - Arquivo não encontrado: t
2025-02-04 18:00:47,786 - WARNING - Arquivo não encontrado: u
2025-02-04 18:00:47,786 - WARNING - Arquivo não encontrado: d
2025-02-04 18:00:47,786 - WARNING - Arquivo não encontrado: o
2025-02-04 18:00:47,787 - WARNING - Arquivo não encontrado: s
2025-02-04 18:00:47,787 - WARNING - Arquivo não encontrado:  
2025-02-04 18:00:47,787 - WARNING - Arquivo não encontrado: e
2025-02-04 18:00:47,788 - WARNING - Arquivo não encontrado:  
2025-02-04 18:00:47,788 - WARNING - Arquivo não encontrado: P
2025-02-04 18:00:47,788 - WARN

Processando o arquivo: C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/sentença\0000124-85.2018.8.26.0540_sentença_unificado.pdf


Processando PDFs:   0%|          | 0/163 [00:00<?, ?it/s]2025-02-04 18:00:47,854 - WARNING - Arquivo não encontrado: C
2025-02-04 18:00:47,855 - WARNING - Arquivo não encontrado: :
2025-02-04 18:00:47,855 - ERROR - Erro ao tentar abrir o PDF /: cannot unpack non-iterable NoneType object
2025-02-04 18:00:47,855 - WARNING - Arquivo não encontrado: E
2025-02-04 18:00:47,855 - WARNING - Arquivo não encontrado: s
2025-02-04 18:00:47,856 - WARNING - Arquivo não encontrado: t
2025-02-04 18:00:47,856 - WARNING - Arquivo não encontrado: u
2025-02-04 18:00:47,856 - WARNING - Arquivo não encontrado: d
2025-02-04 18:00:47,857 - WARNING - Arquivo não encontrado: o
2025-02-04 18:00:47,857 - WARNING - Arquivo não encontrado: s
2025-02-04 18:00:47,857 - WARNING - Arquivo não encontrado:  
2025-02-04 18:00:47,858 - WARNING - Arquivo não encontrado: e
2025-02-04 18:00:47,858 - WARNING - Arquivo não encontrado:  
2025-02-04 18:00:47,858 - WARNING - Arquivo não encontrado: P
2025-02-04 18:00:47,858 - WARN

Nenhum dado extraído para o tipo sentença.


In [ ]:
for pasta in tipos:
    # Pegando os primeiros 5 arquivos PDF de cada tipo de pasta
    source = glob.glob(f"C:/Estudos e Projetos/(2024) Dados LAI MP/separado_doc_tjsp/unificador_documentos/pasta_teste_juncao_docs/{pasta}/*.pdf")
    tipo_doc = pasta
    dados = []  # Lista para armazenar os dados extraídos
    for item in tqdm(source):
        print(item)

In [ ]:
dados